In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
from typing import List
import numpy as np
import pandas as pd
import os
from PIL import Image
from diffusers import DDIMScheduler, UNet2DModel

In [7]:
if torch.backends.mps.is_available():
    device="mps"
elif torch.cuda.is_available():
    device="cuda"
else:
    device="cpu"
device = torch.device(device)

print(f"Device: {device}")

Device: cpu


# Load dataset

In [8]:
# load image -> apply transformations -> dataset of all images
class ImageProcessor:

    def __init__(self, root_dir_path, transformations=None):
        self.root_dir_path  = root_dir_path
        self.transformations = transformations

        self.all_img_paths = [os.path.join(root_dir_path, img_path) for img_path in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self, key):
        img_path = self.all_img_paths[key]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)

        return img

In [9]:
root_dir_path = "dataset/img_align_celeba"
transformations = transforms.Compose([
    transforms.CenterCrop(178), # 178 x 218 => 178 x 178
    transforms.Resize(64), # 178 x 178 => 64 x 64
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Each pixel normalized => [-1, 1]
])
dataset = ImageProcessor(root_dir_path, transformations)

In [10]:
data_loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

# Architecture

### Hyper parameters

In [17]:
timesteps_range = 100
scheduler = DDIMScheduler(
    num_train_timesteps=timesteps_range, 
    beta_start=0.0001, 
    beta_end=0.02, 
)
scheduler.set_timesteps(100)

### Forward Diffusion Process

In [ ]:
class ForwardDiffuser:

    def forward_diffusion_process(self, x, timesteps, scheduler):
        noise = torch.randn_like(x)
        noisy_x = scheduler.add_noise(x, noise, timesteps)
        return noisy_x, noise


### CNN Model (Noise Predictor)

In [11]:
model = UNet2DModel(
    sample_size=64,
    in_channels=3,
    out_channels=3,
    layers_per_block=2,
    down_block_types=(
        "DownBlock2D",  # a regular ResNet downsampling block
        "DownBlock2D",
        "AttnDownBlock2D",  # a ResNet downsampling block with spatial self-attention
        "AttnDownBlock2D",
    ),
    up_block_types=(
        "AttnUpBlock2D",
        "AttnUpBlock2D",  # a ResNet upsampling block with spatial self-attention
        "UpBlock2D",
        "UpBlock2D",  # a regular ResNet upsampling block
    ),
)

### Optimizer

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training

In [ ]:
class Trainer:

    def train(
            self, 
            model: UNet2DModel, 
            criterion: nn.modules.loss._Loss, 
            optimizer: optim.Optimizer, 
            data_loader: DataLoader, 
            forward_diffuser: ForwardDiffuser = ForwardDiffuser(), 
            epochs: int=5,
            timesteps_range: int = 250
        ):

        for epoch in range(epochs):
            for x, _ in data_loader:
                x = x.to(device)
                timesteps = torch.randint(0, timesteps_range, (x.shape[0]), device=device).long()
                (xt, noise) = forward_diffuser.forward_diffusion_process(x, timesteps)
                noise_predicted = model.forward(xt, timesteps)
                loss = criterion(noise_predicted, noise)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            print(f"Epoch {epoch}: Loss {loss.item()}")
            torch.save(model.load_state_dict(), "saved_model/cnn_diffuser.pt")


In [13]:
trainer = Trainer()

In [14]:
# On google collab
#  trainer.train(
#     model,
#     criterion,
#     optimizer,
#     data_loader
# )

# Inference

In [13]:
model.from_pretrained("saved_model")

UNet2DModel(
  (conv_in): Conv2d(3, 224, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (time_proj): Timesteps()
  (time_embedding): TimestepEmbedding(
    (linear_1): Linear(in_features=224, out_features=896, bias=True)
    (act): SiLU()
    (linear_2): Linear(in_features=896, out_features=896, bias=True)
  )
  (down_blocks): ModuleList(
    (0): DownBlock2D(
      (resnets): ModuleList(
        (0-1): 2 x ResnetBlock2D(
          (norm1): GroupNorm(32, 224, eps=1e-05, affine=True, bias=True)
          (conv1): Conv2d(224, 224, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): Linear(in_features=896, out_features=224, bias=True)
          (norm2): GroupNorm(32, 224, eps=1e-05, affine=True, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (conv2): Conv2d(224, 224, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (nonlinearity): SiLU()
        )
      )
      (downsamplers): ModuleList(
        (0): Downsample2D(
     

In [23]:
def inference(model: UNet2DModel, scheduler: DDIMScheduler, image_dim: List[int], timestamps: int = 100):
    with torch.no_grad():
        model.eval()
        x = torch.randn((1, image_dim[0], image_dim[1], image_dim[2]), device=device)

        for timestep in reversed(range(timestamps)):
            timestep_tensor = torch.full((1, ), timestep, device=device, dtype=torch.long)

            with torch.no_grad():
                pred_noise = model(x, timestep_tensor).sample
           
            if timestep > 0:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)

            x = scheduler.step(pred_noise, timestep, x).prev_sample
            
    return x


In [24]:
img = inference(model, scheduler, [3, 64, 64])
# img = (img.clamp(-1, 1) + 1) / 2
torchvision.utils.save_image(img, "generated_image.png")

In [18]:
img

tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 1., 1., 1.],
          [0., 0., 0.,  ..., 1., 1., 1.],
          [0., 0., 0.,  ..., 1., 1., 1.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 1., 1., 1.],
          [0., 0., 0.,  ..., 1., 1., 1.],
          [0., 0., 0.,  ..., 1., 1., 1.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 1., 1., 1.],
          [0., 0., 0.,  ..., 1., 1., 1.],
          [0., 0., 0.,  ..., 1., 1., 1.]]]])